# Aishani HTTP 이상 탐지: 원본 방식 재현과 측정

**CPU 런타임 / 하드웨어 가속기 없음**으로 전체 실행하세요. GPU/TPU가 필요 없습니다.
실용 최소 추정은 CPU 2 cores / RAM 4 GB, 권장은 RAM 8 GB입니다. 원본 2.4 MB 자료 기준이며 Colab 자원은 고정 보장이 아닙니다.

이 노트북은 IP 전체 기간의 7개 피처, 전체 자료 MinMaxScaler, IF·OCSVM·KMeans의 원본 설정을 유지합니다.
원본 로그에는 공격 label이 없어 **탐지 정확도를 측정할 수 없습니다**. 모델이 이상으로 표시한 수와 실행 시간·프로세스 peak RSS를 측정합니다.
같은 자료를 학습·판정하는 원본 실험이며, 개선 모델의 holdout 성능과 직접 비교하지 않습니다.

[원본 고정 commit](https://github.com/Aishani2001/VMWare-API-Misuse-or-Abuse/tree/e8c5fc3fd149985ef73a59df3f044368d607e5d6)
· [Colab 자원 안내](https://research.google.com/colaboratory/faq.html)

공개 자료만 사용합니다. 인증정보·실사용자의 로그를 업로드하지 마세요. Drive 연결 없이 실행됩니다.


## 1. 고정 의존성 설치
새 런타임에서 시작하세요. 이미 로드한 numpy 등과 설치 버전이 다르면 설치 후 런타임을 다시 시작하세요.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path("/content/aishani_http_baseline")
ROOT.mkdir(parents=True, exist_ok=True)
requirements = """numpy==2.2.6
scikit-learn==1.7.2
scipy==1.15.3
joblib==1.5.2
threadpoolctl==3.6.0
"""
(ROOT / "requirements.txt").write_text(requirements)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(ROOT / "requirements.txt")], check=True)
print("Dependencies installed. Use a fresh CPU runtime; restart if versions were already imported.")


## 2. 독립 구현과 검증 코드
원본 소스 복사가 아닌 알고리즘 재현입니다. 첫 간격 0·active-day 평균·누락 바이트 0·ddof=1을 유지합니다. KMeans n_init=10을 명시했습니다. elbow/t-SNE 그래프는 생략했습니다.

In [ ]:
%%writefile /content/aishani_http_baseline/baseline.py
"""Independent reproduction of Aishani's unlabeled, in-sample HTTP experiment.

No service connections or network calls. Only public/sanitized Apache access logs.
The historical aggregation choices are deliberately preserved for comparison.
"""
from __future__ import annotations

import argparse
import csv
from datetime import datetime
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import re
import resource
import statistics
import sys
import time

import joblib
import numpy as np
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import OneClassSVM
from threadpoolctl import threadpool_limits

REFERENCE_COMMIT = "e8c5fc3fd149985ef73a59df3f044368d607e5d6"
REFERENCE_URL = "https://github.com/Aishani2001/VMWare-API-Misuse-or-Abuse"
DATA_SHA256 = "f15c31e905f86c7b4b6ab44aee74d0a2086dce89f010187d983edea7ef0364ef"
GOLDEN_SHA256 = "3caacb4eddff809ecb5d9df606431534441f191e8287f831274074089725f793"
VERSION = "aishani-http-reference/1"
FEATURES = (
    "Total Requests", "Daily Mean", "GET requests", "Successful requests",
    "Mean Return Size", "Return Size Std", "Mean Time Difference",
)
# Preserve the original IPv4-like matching and silently skipped non-matches.
LOG_PATTERN = re.compile(r'([\d\.]+) - - \[(.*?)\] "(.*?)" (\d+) (.+) "(.*?)" "(.*?)"')


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def subject_id(ip: str) -> str:
    """Stable public-data join key; this is not a privacy guarantee for real IPs."""
    return hashlib.sha256((VERSION + "\0" + ip).encode()).hexdigest()


def aggregate_lines(lines) -> tuple[list[str], np.ndarray, dict]:
    by_ip: dict[str, list[tuple]] = {}
    quality = {"lines": 0, "parsed": 0, "skipped": 0, "missing_bytes_mapped_to_zero": 0}
    for line in lines:
        quality["lines"] += 1
        match = LOG_PATTERN.match(line)
        if match is None:
            quality["skipped"] += 1
            continue
        ip, timestamp, request, status, size, _referer, _ua = match.groups()
        try:
            # Original notebook discards the numeric timezone offset.
            stamp = datetime.strptime(timestamp.split(" ")[0], "%d/%b/%Y:%H:%M:%S")
            missing = "-" in size
            returned = 0 if missing else int(size)
            success = int(200 <= int(status) <= 299)
        except ValueError:
            raise ValueError(f"Invalid matched record at line {quality['lines']}; raw values omitted") from None
        quality["missing_bytes_mapped_to_zero"] += int(missing)
        quality["parsed"] += 1
        by_ip.setdefault(ip, []).append((stamp, returned, int("GET" in request[:4]), success))
    if not by_ip:
        raise ValueError("No records matched the historical IPv4 access-log parser")
    ips = sorted(by_ip)
    rows = []
    for ip in ips:
        records = sorted(by_ip[ip], key=lambda item: item[0])
        day_counts: dict = {}
        gaps = []
        previous = None
        for stamp, _size, _get, _success in records:
            day = stamp.date()
            day_counts[day] = day_counts.get(day, 0) + 1
            gap = 0 if previous is None or previous.date() != day else (stamp - previous).seconds
            gaps.append(gap)
            previous = stamp
        sizes = [r[1] for r in records]
        rows.append([
            len(records), len(records) / len(day_counts),
            sum(r[2] for r in records), sum(r[3] for r in records),
            statistics.mean(sizes), statistics.stdev(sizes) if len(sizes) > 1 else 0,
            statistics.mean(gaps),
        ])
    return ips, np.asarray(rows, dtype=np.float64), quality


def check_reference(ips: list[str], matrix: np.ndarray, golden_path: Path) -> dict:
    """Compare every feature against the author's CSV, without emitting raw IPs."""
    if sha256_file(golden_path) != GOLDEN_SHA256:
        raise ValueError("Reference CSV checksum mismatch")
    with golden_path.open(newline="", encoding="utf-8") as source:
        expected = {row["IP"]: [float(row[name]) for name in FEATURES] for row in csv.DictReader(source)}
    if set(ips) != set(expected):
        raise ValueError("Reference subject set mismatch")
    target = np.asarray([expected[ip] for ip in ips])
    if not np.allclose(matrix, target, rtol=1e-10, atol=1e-9):
        raise ValueError("Feature reproduction differs from reference CSV")
    return {
        "passed": True, "subjects": len(ips), "feature_values": matrix.size,
        "max_absolute_difference": float(np.max(np.abs(matrix - target))),
        "rtol": 1e-10, "atol": 1e-9, "reference_csv_sha256": GOLDEN_SHA256,
    }


def peak_rss_bytes() -> int:
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return int(peak if sys.platform == "darwin" else peak * 1024)


def machine_info() -> dict:
    try:
        ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")
    except (ValueError, OSError):
        ram = None
    return {
        "platform": platform.platform(), "machine": platform.machine(),
        "cpu_count": os.cpu_count(), "joblib_effective_cpu_count": joblib.cpu_count(),
        "loky_cpu_limit": os.environ.get("LOKY_MAX_CPU_COUNT"), "physical_ram_bytes": ram,
        "python": platform.python_version(),
        "dependencies": {name: importlib.metadata.version(name) for name in
                         ("numpy", "scikit-learn", "scipy", "joblib", "threadpoolctl")},
    }


def run(log_path: Path, output: Path, golden_path: Path | None = None,
        models: tuple[str, ...] = ("iforest", "ocsvm", "kmeans"), threads: int = 2) -> dict:
    if threads < 1 or not models or len(set(models)) != len(models):
        raise ValueError("Positive threads and distinct models are required")
    if any(name not in ("iforest", "ocsvm", "kmeans") for name in models):
        raise ValueError("Unknown model")
    # Never overwrite an earlier run or its model artifacts.
    if output.exists():
        raise FileExistsError("Run output already exists; use a new directory")
    started = time.perf_counter()
    data_hash = sha256_file(log_path)
    stage = time.perf_counter()
    with log_path.open(encoding="utf-8") as source:
        ips, matrix, quality = aggregate_lines(source)
    features_s = time.perf_counter() - stage
    golden = None
    if golden_path is not None:
        if data_hash != DATA_SHA256:
            raise ValueError("Golden comparison requires the pinned public reference log")
        golden = check_reference(ips, matrix, golden_path)
    scaler = MinMaxScaler()
    stage = time.perf_counter()
    scaled = scaler.fit_transform(matrix)
    scaling_s = time.perf_counter() - stage
    summaries = {}
    estimators = {}
    scores_by_model = {}
    flags_by_model = {}
    with threadpool_limits(limits=threads):
        for name in models:
            if name == "iforest":
                params = dict(n_jobs=-1, n_estimators=200, max_features=3,
                              random_state=42, contamination=0.01)
                estimator = IsolationForest(**params)
                fit_data = scaled
                threshold = 0.0
                score_definition = "negative decision_function; larger is more anomalous"
            elif name == "ocsvm":
                params = dict(nu=0.01, degree=2, kernel="rbf", gamma="scale")
                estimator = OneClassSVM(**params)
                cutoff = float(np.quantile(matrix[:, 0], .99))
                fit_data = scaled[matrix[:, 0] <= cutoff]
                threshold = 0.0
                score_definition = "negative decision_function; larger is more anomalous"
            else:
                if len(ips) < 5:
                    raise ValueError("KMeans reproduction requires at least five subjects")
                params = dict(n_clusters=5, random_state=42, n_init=10)
                estimator = KMeans(**params)
                fit_data = scaled
                threshold = 4.7
                score_definition = "sum of squared distances to ALL five centroids"
            stage = time.perf_counter()
            estimator.fit(fit_data)
            fit_s = time.perf_counter() - stage
            stage = time.perf_counter()
            if name == "kmeans":
                scores = np.square(estimator.transform(scaled)).sum(axis=1)
                flags = scores >= threshold
            else:
                scores = -estimator.decision_function(scaled)
                flags = estimator.predict(scaled) == -1
            predict_s = time.perf_counter() - stage
            scores_by_model[name] = scores
            flags_by_model[name] = flags
            estimators[name] = estimator
            summaries[name] = {
                "params": params, "training_subjects": len(fit_data),
                "scoring_subjects": len(ips), "flagged_subjects": int(flags.sum()),
                "flag_rate": float(flags.mean()), "threshold": threshold,
                "score_definition": score_definition, "fit_seconds": fit_s,
                "score_and_predict_seconds": predict_s,
                "score_quantiles": dict(zip(("min", "p50", "p95", "p99", "max"),
                                            np.quantile(scores, (0, .5, .95, .99, 1)).tolist())),
            }
            if name == "ocsvm":
                summaries[name]["training_request_count_p99"] = cutoff
    pipeline_s = time.perf_counter() - started
    output.mkdir(parents=True, exist_ok=False)
    header = ["subject_id", *FEATURES]
    for name in models:
        header.extend((name + "_score", name + "_flag"))
    with (output / "predictions.csv").open("w", newline="", encoding="utf-8") as destination:
        writer = csv.writer(destination)
        writer.writerow(header)
        for index, ip in enumerate(ips):
            row = [subject_id(ip), *matrix[index].tolist()]
            for name in models:
                row.extend((float(scores_by_model[name][index]), int(flags_by_model[name][index])))
            writer.writerow(row)
    artifact = output / "baseline.joblib"
    joblib.dump({"version": VERSION, "features": FEATURES, "scaler": scaler,
                 "estimators": estimators, "reference_commit": REFERENCE_COMMIT,
                 "input_sha256": data_hash, "mode": "reference_reproduction_in_sample",
                 "thresholds": {name: summaries[name]["threshold"] for name in models},
                 "score_definitions": {name: summaries[name]["score_definition"] for name in models}},
                artifact, compress=3)
    report = {
        "version": VERSION, "mode": "reference_reproduction_in_sample",
        "reference": {"url": REFERENCE_URL, "commit": REFERENCE_COMMIT},
        "code_sha256": sha256_file(Path(__file__)), "seed": 42,
        "input_sha256": data_hash, "input_bytes": log_path.stat().st_size,
        "unit": "one IP over the full input period", "subjects": len(ips),
        "features": list(FEATURES), "quality": quality, "golden_check": golden,
        "environment": machine_info(), "blas_openmp_thread_limit": threads,
        "isolation_forest_jobs": "all visible CPU cores (n_jobs=-1)",
        "models": summaries,
        "detection_metrics": {"precision": None, "recall": None, "f1": None,
                              "pr_auc": None, "fpr": None,
                              "reason": "No ground-truth attack labels; flag_rate is not FPR"},
        "timing": {"feature_seconds": features_s, "scaling_seconds": scaling_s,
                   "pipeline_seconds_excluding_export": pipeline_s,
                   "end_to_end_seconds_excluding_report_write": time.perf_counter() - started},
        "memory": {"process_lifetime_peak_rss_bytes": peak_rss_bytes(),
                   "scope": "current process including imports; not incremental dataset memory"},
        "artifacts": {path.name: {"sha256": sha256_file(path), "bytes": path.stat().st_size}
                      for path in (artifact, output / "predictions.csv")},
        "reproduction_limits": [
            "Original dependency versions were not locked; model outputs are version dependent",
            "KMeans historical default n_init=10 is explicit; modern default auto is not used",
            "Elbow sweep and t-SNE figures are omitted; they do not enter anomaly scoring",
            "Only algorithmic features are reproduced, not original dataframe/raw CSV memory use",
            "No held-out accuracy or token-theft detection claim follows from this experiment",
        ],
    }
    (output / "report.json").write_text(json.dumps(report, indent=2, allow_nan=False) + "\n")
    return report


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--log", required=True, type=Path)
    parser.add_argument("--output", required=True, type=Path)
    parser.add_argument("--reference-csv", type=Path)
    parser.add_argument("--models", nargs="+", choices=("iforest", "ocsvm", "kmeans"),
                        default=("iforest", "ocsvm", "kmeans"))
    parser.add_argument("--threads", type=int, default=2)
    args = parser.parse_args()
    report = run(args.log, args.output, args.reference_csv, tuple(args.models), args.threads)
    print(json.dumps({"mode": report["mode"], "subjects": report["subjects"],
                      "quality": report["quality"], "golden_check": report["golden_check"],
                      "models": report["models"], "timing": report["timing"],
                      "memory": report["memory"]}, indent=2, allow_nan=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/aishani_http_baseline/test_baseline.py
"""Historical aggregation semantics, independent fixtures and artifact checks."""
from pathlib import Path
import tempfile
import unittest

import numpy as np

import baseline


def line(ip, date, size, method="GET", status=200, timezone="+0000"):
    return f'{ip} - - [{date} {timezone}] "{method} /fixture HTTP/1.1" {status} {size} "-" "fixture"\n'


class FeaturesTest(unittest.TestCase):
    def test_active_days_first_zero_and_sample_std(self):
        logs = [line("10.0.0.1", "01/Jan/2020:00:00:10", "300", "POST", 404),
                line("10.0.0.1", "03/Jan/2020:00:00:00", "-"),
                line("10.0.0.1", "01/Jan/2020:00:00:00", "100"), "invalid\n"]
        _, matrix, quality = baseline.aggregate_lines(logs)
        # Active days=2; gaps are [0,10,0]; std uses ddof=1, missing bytes=0.
        np.testing.assert_allclose(matrix[0], [3, 1.5, 2, 2, 400 / 3,
                                             np.std([100, 300, 0], ddof=1), 10 / 3])
        self.assertEqual(quality, {"lines": 4, "parsed": 3, "skipped": 1,
                                   "missing_bytes_mapped_to_zero": 1})

    def test_singleton_timezone_discarded_and_order(self):
        logs = [line("10.0.0.2", "01/Jan/2020:00:00:00", "2"),
                line("10.0.0.1", "01/Jan/2020:00:00:10", "4", timezone="+0900"),
                line("10.0.0.1", "01/Jan/2020:00:00:00", "4", timezone="-0500")]
        ips, matrix, _ = baseline.aggregate_lines(logs)
        np.testing.assert_allclose(matrix[0], [2, 2, 2, 2, 4, 0, 5])
        np.testing.assert_allclose(matrix[1], [1, 1, 1, 1, 2, 0, 0])
        other_ips, other, _ = baseline.aggregate_lines(reversed(logs))
        self.assertEqual(ips, other_ips)
        np.testing.assert_array_equal(matrix, other)

    def test_no_matching_records_fail(self):
        with self.assertRaises(ValueError):
            baseline.aggregate_lines(["invalid\n"])

    def test_malformed_matching_record_omits_raw_values(self):
        raw = line("10.0.0.1", "not-a-date", "private-value")
        with self.assertRaisesRegex(ValueError, "raw values omitted") as raised:
            baseline.aggregate_lines([raw])
        self.assertNotIn("private-value", str(raised.exception))

    def test_raw_identity_not_exported_and_deterministic_flags(self):
        with tempfile.TemporaryDirectory() as name:
            root = Path(name)
            path = root / "public.log"
            path.write_text("".join(line(f"10.0.0.{i}", "01/Jan/2020:00:00:00", str(i * 10))
                                    for i in range(1, 15)))
            first = baseline.run(path, root / "one", models=("iforest",))
            second = baseline.run(path, root / "two", models=("iforest",))
            predictions = (root / "one" / "predictions.csv").read_text()
            self.assertNotIn("10.0.0.", predictions)
            self.assertEqual(predictions, (root / "two" / "predictions.csv").read_text())
            self.assertEqual(first["detection_metrics"]["fpr"], None)
            self.assertEqual(first["models"]["iforest"]["flagged_subjects"],
                             second["models"]["iforest"]["flagged_subjects"])
            with self.assertRaises(FileExistsError):
                baseline.run(path, root / "one")


if __name__ == "__main__":
    unittest.main()


## 3. 고정 공개 자료 다운로드
commit과 SHA-256을 확인합니다. 원문 로그는 출력하지 않습니다.

In [ ]:
import hashlib
import urllib.request

COMMIT = "e8c5fc3fd149985ef73a59df3f044368d607e5d6"
BASE = f"https://raw.githubusercontent.com/Aishani2001/VMWare-API-Misuse-or-Abuse/{COMMIT}/"
FILES = {
    "access2.log": "f15c31e905f86c7b4b6ab44aee74d0a2086dce89f010187d983edea7ef0364ef",
    "finaldata.csv": "3caacb4eddff809ecb5d9df606431534441f191e8287f831274074089725f793",
}
for name, expected in FILES.items():
    with urllib.request.urlopen(BASE + name, timeout=60) as response:
        data = response.read(5_000_001)
    if len(data) > 5_000_000 or hashlib.sha256(data).hexdigest() != expected:
        raise ValueError("Public reference checksum/size mismatch: " + name)
    (ROOT / name).write_bytes(data)
    print(name, len(data), "bytes; checksum verified")
# Do not display raw IPs, URLs, user agents or request records.


## 4. 독립 fixture 검증

In [ ]:
test = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", str(ROOT), "-v"],
    capture_output=True, text=True,
)
print(test.stdout)
print(test.stderr)
test.check_returncode()


## 5. 원본 CSV 대조 후 새 프로세스에서 3회 측정
1,753개 IP × 7개 값 = 12,271개 피처 값을 원본 CSV와 대조합니다. seed=42로 고정합니다. 모델 fit/predict 시간은 프로세스 import와 설치·다운로드 시간을 제외합니다.

In [ ]:
import importlib.metadata
import json
import os
from datetime import datetime, timezone

expected = {line.split("==")[0]: line.split("==")[1]
            for line in requirements.splitlines() if "==" in line}
for package, version in expected.items():
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)

run_root = ROOT / "runs" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
reports = []
child_env = dict(os.environ, LOKY_MAX_CPU_COUNT="2")
for repetition in range(1, 4):
    output = run_root / f"run-{repetition}"
    result = subprocess.run([
        sys.executable, str(ROOT / "baseline.py"),
        "--log", str(ROOT / "access2.log"),
        "--reference-csv", str(ROOT / "finaldata.csv"),
        "--output", str(output), "--threads", "2",
    ], capture_output=True, text=True, env=child_env)
    if result.returncode:
        print(result.stderr)
        result.check_returncode()
    report = json.loads((output / "report.json").read_text())
    reports.append(report)
    print(f"Run {repetition}: golden={report['golden_check']['passed']}, "
          f"{report['timing']['pipeline_seconds_excluding_export']:.3f}s pipeline, "
          f"{report['memory']['process_lifetime_peak_rss_bytes'] / 2**20:.1f} MiB peak RSS")


## 6. 측정 요약
flag rate는 오탐률(FPR)이 아닙니다. peak RSS는 import를 포함한 하위 프로세스 생애 최대 메모리이며 notebook 전체 메모리가 아닙니다.

In [ ]:
import statistics

# Compare deterministic outcomes, then summarize timing repeats (not accuracy repeats).
for report in reports[1:]:
    for name in reports[0]["models"]:
        if report["models"][name]["flagged_subjects"] != reports[0]["models"][name]["flagged_subjects"]:
            raise RuntimeError("Flag count changed across repeats: " + name)
summary = {
    "environment_label": "Colab CPU; confirm actual environment below",
    "environment": reports[0]["environment"],
    "mode": reports[0]["mode"], "repetitions": len(reports), "seed": 42,
    "input_sha256": reports[0]["input_sha256"], "input_bytes": reports[0]["input_bytes"],
    "code_sha256": reports[0]["code_sha256"], "quality": reports[0]["quality"],
    "subjects": reports[0]["subjects"], "golden_check": reports[0]["golden_check"],
    "blas_openmp_thread_limit": reports[0]["blas_openmp_thread_limit"],
    "models": {}, "detection_metrics": reports[0]["detection_metrics"],
    "pipeline_seconds_median": statistics.median(r["timing"]["pipeline_seconds_excluding_export"] for r in reports),
    "end_to_end_seconds_excluding_report_write_median": statistics.median(r["timing"]["end_to_end_seconds_excluding_report_write"] for r in reports),
    "process_lifetime_peak_rss_bytes_max": max(r["memory"]["process_lifetime_peak_rss_bytes"] for r in reports),
}
for name, model in reports[0]["models"].items():
    summary["models"][name] = {
        "flagged_subjects": model["flagged_subjects"], "flag_rate": model["flag_rate"],
        "fit_seconds_median": statistics.median(r["models"][name]["fit_seconds"] for r in reports),
        "score_and_predict_seconds_median": statistics.median(r["models"][name]["score_and_predict_seconds"] for r in reports),
    }
(run_root / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print("Environment:", summary["environment"])
print("Model      Flagged IPs   Fit (median s)   Score + predict (median s)")
for name, model in summary["models"].items():
    print(f"{name:10} {model['flagged_subjects']:11} {model['fit_seconds_median']:16.4f} {model['score_and_predict_seconds_median']:26.4f}")
print("No ground truth: precision/recall/F1/PR-AUC/FPR are unavailable.")
print("3 runs use seed=42. These repeats measure runtime, not accuracy uncertainty.")


## 7. 결과 저장
ZIP을 로컬에 저장하세요. 향후 같은 label 자료와 평가 단위, split을 고정한 뒤 원본 방식의 heldout 평가와 개선 모델을 비교합니다. 모델은 신뢰한 자기 산출물과 checksum만 확인해서 로드하세요.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(str(run_root) + "-results", "zip", root_dir=run_root)
files.download(zip_path)
# Contains report.json, pseudonymous predictions.csv and baseline.joblib for each run.
# Public raw access log is deliberately excluded. Save results before runtime ends.
